# Bridging AI forecasts and fuzzy optimization: reproducible notebook

Runs the whole pipeline from the raw Olist CSV files to every number, table and simulation in the paper:

1. data cleaning (sports subset, weekly demand, defect proxy)
2. price sensitivity (within-product elasticity) and the calibrated linear demand $D(P,\tau)=a_0-a_1P+a_2\tau$
3. Stage-1 models (demand benchmark, dissatisfaction classifier) and prediction intervals
4. crisp and fuzzy optimization of TPU(Q, B, P), with the price restricted to the observed range
5. paired Monte Carlo comparison of the crisp and fuzzy policies
6. sensitivity tables and numerical concavity checks, signal tests

**How to use:** download the Olist dataset from Kaggle (`olistbr/brazilian-ecommerce`), put the CSV (or zip) files in one folder,
set `DATA_DIR` in the setup cell, and run **Kernel > Restart & Run All** (a few minutes). On Google Colab run the optional upload cell first.
Everything is saved to `OUT_DIR` (`results.json`, `weekly.csv`).

## 0. Setup

**In plain words.** This cell block reads the settings (folders, random seed 42) and defines `R`, a dictionary in which every number used in the paper is stored; at the end `R` is written to `outputs/results.json`. The helper `find(key)` looks for a CSV whose name contains `key` anywhere inside `DATA_DIR`, and unzips archives if needed.

In [1]:
# --- OPTIONAL (Google Colab only): upload the 5 Olist files (csv or zip) into ./olist_data ---
# from google.colab import files
# import os; os.makedirs("olist_data", exist_ok=True)
# up = files.upload()
# for name, content in up.items(): open(f"olist_data/{name}", "wb").write(content)

In [2]:
import os, glob, zipfile, json, warnings
import numpy as np, pandas as pd
from scipy import stats
from scipy.optimize import minimize, minimize_scalar
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingClassifier
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import (KFold, StratifiedGroupKFold, TimeSeriesSplit, cross_val_predict)
from sklearn.metrics import r2_score, mean_absolute_error, f1_score, roc_auc_score
warnings.filterwarnings("ignore")

DATA_DIR = "olist_data"      # <-- folder that contains the Olist csv (or zip) files
OUT_DIR = "outputs"          # results are written here
SEED = 42
os.makedirs(OUT_DIR, exist_ok=True)
R = {}                       # every number used in the paper is collected here and saved at the end

for z in glob.glob(f"{DATA_DIR}/*.zip"):
    zipfile.ZipFile(z).extractall(DATA_DIR)

def find(key):
    hits = sorted(glob.glob(f"{DATA_DIR}/**/*{key}*.csv", recursive=True))
    if not hits: raise FileNotFoundError(f"No csv containing '{key}' in {DATA_DIR}")
    return hits[0]

print("pandas", pd.__version__, "| numpy", np.__version__)

pandas 2.3.3 | numpy 2.3.5


## 1. Data cleaning (sports subset)

**What happens here.** (1) The order, order-item, product and review tables are merged and only the sports categories are kept. (2) Some orders have several reviews; they are averaged to one score per order, otherwise the same order-line would be counted more than once. (3) Only delivered orders are kept. (4) *Weekly demand* is the number of order-lines sold in each week, and the weekly average price is the mean price of those lines. (5) The *defect proxy* is a line whose order got a review score of 2 or lower; this is dissatisfaction, not a measured defect. The share of such lines is `theta_hat`.

In [3]:
orders = pd.read_csv(find("olist_orders_dataset"),
                     parse_dates=["order_purchase_timestamp", "order_delivered_customer_date",
                                  "order_estimated_delivery_date"])
items = pd.read_csv(find("olist_order_items_dataset"))
prods = pd.read_csv(find("olist_products_dataset"))
rev = pd.read_csv(find("olist_order_reviews_dataset"))
tr = pd.read_csv(find("product_category_name_translation"))

prods = prods.merge(tr, on="product_category_name", how="left")
sp = prods[prods["product_category_name_english"].isin(["sports_leisure", "fashion_sport"])]
lines = items.merge(sp[["product_id", "product_category_name_english"]], on="product_id").merge(orders, on="order_id")
R["n_lines_all"] = len(lines)
lines = lines[lines["order_status"] == "delivered"].copy()
R["n_lines_delivered"] = len(lines)

# ONE review score per order (orders with several reviews would otherwise duplicate their lines)
rs = rev.groupby("order_id")["review_score"].mean().rename("review_score").reset_index()
lines = lines.merge(rs, on="order_id", how="left")
lines["delay_days"] = (lines["order_delivered_customer_date"]
                       - lines["order_estimated_delivery_date"]).dt.total_seconds() / 86400
lines["week"] = lines["order_purchase_timestamp"].dt.to_period("W").dt.start_time
print("sports order-lines (all statuses):", R["n_lines_all"], "| delivered:", R["n_lines_delivered"],
      "| span:", lines["order_purchase_timestamp"].min().date(), "->", lines["order_purchase_timestamp"].max().date())

# weekly demand (units = order-lines) over weeks that contain at least one order
wk = (lines.groupby("week").agg(demand=("price", "size"), avg_price=("price", "mean"))
      .reset_index().sort_values("week").reset_index(drop=True))
wk["t_obs"] = np.arange(1, len(wk) + 1)
wk["month"] = wk["week"].dt.month
wk["peak"] = wk["month"].isin([11, 12]).astype(int)
wk["jan"] = (wk["month"] == 1).astype(int)
R.update(n_weeks=len(wk), mean_demand=float(wk.demand.mean()), min_demand=int(wk.demand.min()), max_demand=int(wk.demand.max()),
         price_min=float(wk.avg_price.min()), price_p10=float(wk.avg_price.quantile(.1)), price_median=float(wk.avg_price.median()),
         price_mean=float(wk.avg_price.mean()), price_p90=float(wk.avg_price.quantile(.9)), price_max=float(wk.avg_price.max()),
         nov_mean_demand=float(wk[wk.month == 11].demand.mean()))
print("weekly observations:", R["n_weeks"], "| mean demand %.1f (range %d-%d)" % (R["mean_demand"], R["min_demand"], R["max_demand"]),
      "| weekly avg price %.1f-%.1f (mean %.1f)" % (R["price_min"], R["price_max"], R["price_mean"]))

# defect proxy: line-level, review score <= 2 ; needs review and delivery date
Ln = lines.dropna(subset=["review_score", "delay_days"]).copy()
Ln["dissat"] = (Ln["review_score"] <= 2).astype(int)
R.update(n_lines_with_review=int(lines.review_score.notna().sum()), n_def=len(Ln), n_def_orders=int(Ln.order_id.nunique()),
         k_def=int(Ln.dissat.sum()), theta_hat=float(Ln.dissat.mean()))
print("defect-proxy sample: n=%d lines, %d orders, k=%d dissatisfied, theta_hat=%.4f"
      % (R["n_def"], R["n_def_orders"], R["k_def"], R["theta_hat"]))
wk.to_csv(f"{OUT_DIR}/weekly.csv", index=False)

sports order-lines (all statuses): 8671 | delivered: 8460 | span: 2016-10-03 -> 2018-08-29
weekly observations: 88 | mean demand 96.1 (range 3-222) | weekly avg price 46.6-194.3 (mean 112.5)
defect-proxy sample: n=8406 lines, 7511 orders, k=1093 dissatisfied, theta_hat=0.1300


## 2. Price sensitivity and the calibrated demand function

The weekly average price mixes products (expensive products sell more), so a plain regression of weekly demand on weekly price is confounded:
its price coefficient is **positive** and not significant. Price sensitivity is therefore estimated from *within-product* price variation
(log-log regression with product fixed effects on a product-week panel, standard errors clustered by product).
The linear model $D=a_0-a_1P+a_2\tau$ is then calibrated: $a_1=-\varepsilon\,\bar D/\bar P$ (slope of the constant-elasticity curve at the sample means),
and $a_0,a_2$ come from OLS of $D+a_1(P-\bar P)$ on the trend index.

**What happens here.** (a) A plain regression of weekly demand on the weekly average price gives a *positive* price coefficient. That is not price sensitivity: expensive products sell more, and the weekly average mixes products. (b) Therefore the price effect is estimated *within* each product (log-log regression with product fixed effects, standard errors clustered by product). (c) The elasticity is turned into the slope `a1 = -elasticity * mean demand / mean price` of the linear demand curve, and `a0`, `a2` are refitted by least squares with `a1` held fixed.

In [4]:
y = wk["demand"].values; Pbar = wk.avg_price.mean(); Dbar = y.mean()

# (a) naive weekly OLS (for the record): shows the confounded positive price coefficient
Xn = np.column_stack([np.ones(len(wk)), wk.avg_price, wk.t_obs])
bn, *_ = np.linalg.lstsq(Xn, y, rcond=None)
res_n = y - Xn @ bn; se_n = np.sqrt(np.diag(res_n @ res_n / (len(y) - 3) * np.linalg.inv(Xn.T @ Xn)))
R["naive_price_coef"] = float(bn[1]); R["naive_price_p"] = float(2 * (1 - stats.t.cdf(abs(bn[1] / se_n[1]), len(y) - 3)))
R["naive_r2"] = float(1 - res_n @ res_n / ((y - Dbar) @ (y - Dbar)))
print(f"naive weekly OLS: price coef = {bn[1]:+.4f} (p={R['naive_price_p']:.2f}), trend = {bn[2]:+.4f}, R2={R['naive_r2']:.3f}")

# (b) within-product elasticity: product-week panel, product fixed effects, clustered SE
pw = lines.groupby(["product_id", "week"]).agg(q=("price", "size"), price=("price", "mean")).reset_index()
pw = pw[pw.groupby("product_id")["week"].transform("nunique") >= 4].copy()
pw["lq"] = np.log(pw.q); pw["lp"] = np.log(pw.price)
for c_ in ["lq", "lp"]: pw[c_ + "_d"] = pw[c_] - pw.groupby("product_id")[c_].transform("mean")
xx = pw.lp_d.values; yy = pw.lq_d.values
eps_hat = (xx @ yy) / (xx @ xx); u = yy - eps_hat * xx
G = pw.product_id.nunique(); N = len(pw)
sc = pd.Series(xx * u).groupby(pw.product_id.values).sum().values
se_eps = np.sqrt(G / (G - 1) * (N - 1) / (N - G - 1) * (sc ** 2).sum()) / (xx @ xx)
R.update(eps=float(eps_hat), eps_se=float(se_eps), panel_products=int(G), panel_rows=int(N))
print(f"within-product elasticity: {eps_hat:.3f} (clustered se {se_eps:.3f}, t={eps_hat / se_eps:.1f}); {G} products, {N} product-weeks")

# (c) calibrated linear demand  D = a0 - a1*P + a2*tau
def calibrate(a1_, idx=None):
    idx = np.arange(len(wk)) if idx is None else idx
    yadj = y[idx] + a1_ * (wk.avg_price.values[idx] - Pbar)
    Xc = np.column_stack([np.ones(len(idx)), wk.t_obs.values[idx]])
    (alpha, a2_), *_ = np.linalg.lstsq(Xc, yadj, rcond=None)
    return alpha + a1_ * Pbar, a2_                  # a0, a2
def predict(a0_, a1_, a2_, idx): return a0_ - a1_ * wk.avg_price.values[idx] + a2_ * wk.t_obs.values[idx]

a1 = -eps_hat * Dbar / Pbar
a0, a2 = calibrate(a1)
fit = predict(a0, a1, a2, np.arange(len(wk)))
R.update(a0=float(a0), a1=float(a1), a2=float(a2), r2_in=float(1 - ((y - fit) ** 2).sum() / ((y - Dbar) ** 2).sum()))
print(f"calibrated demand: a0={a0:.3f}  a1={a1:.4f}  a2={a2:.4f}   (in-sample R2={R['r2_in']:.3f})")

naive weekly OLS: price coef = +0.1855 (p=0.34), trend = +1.1799, R2=0.403
within-product elasticity: -0.245 (clustered se 0.151, t=-1.6); 421 products, 3229 product-weeks
calibrated demand: a0=67.574  a1=0.2096  a2=1.1715   (in-sample R2=0.373)


### 2b. Out-of-sample checks of the demand model, random-forest benchmark, residual quantiles

**What happens here.** The calibrated demand model is tested on data it has not seen: (i) 5-fold cross-validation with shuffled weeks (optimistic, because demand trends upward) and (ii) forward-chaining, which always predicts later weeks from earlier ones (the honest test). A random forest is fitted only as a benchmark. The out-of-fold residuals give the quantiles `RQ` (5%, 25%, 75%, 95%) that become the breakpoints of the fuzzy demand.

In [5]:
cv = KFold(5, shuffle=True, random_state=SEED)
oof = np.zeros(len(wk))
for tr_i, te_i in cv.split(wk):
    a0_, a2_ = calibrate(a1, tr_i); oof[te_i] = predict(a0_, a1, a2_, te_i)
R["cv_r2"] = float(r2_score(y, oof)); R["cv_mae"] = float(mean_absolute_error(y, oof))
pr_, tr_ = [], []
for tr_i, te_i in TimeSeriesSplit(5).split(wk):
    a0_, a2_ = calibrate(a1, tr_i); pr_ += list(predict(a0_, a1, a2_, te_i)); tr_ += list(y[te_i])
R["fc_r2"] = float(r2_score(tr_, pr_)); R["fc_mae"] = float(mean_absolute_error(tr_, pr_))
print(f"calibrated linear model: shuffled 5-fold CV R2={R['cv_r2']:.3f} (MAE {R['cv_mae']:.1f}) | forward-chaining R2={R['fc_r2']:.3f} (MAE {R['fc_mae']:.1f})")

rf = RandomForestRegressor(n_estimators=300, random_state=SEED)
feats_rf = ["t_obs", "avg_price", "month", "peak", "jan"]
p_rf = cross_val_predict(rf, wk[feats_rf], y, cv=cv)
R["rf_cv_r2"] = float(r2_score(y, p_rf)); R["rf_cv_mae"] = float(mean_absolute_error(y, p_rf))
pr_, tr_ = [], []
for tr_i, te_i in TimeSeriesSplit(5).split(wk):
    rf.fit(wk.iloc[tr_i][feats_rf], y[tr_i]); pr_ += list(rf.predict(wk.iloc[te_i][feats_rf])); tr_ += list(y[te_i])
R["rf_fc_r2"] = float(r2_score(tr_, pr_))
print(f"random-forest benchmark: shuffled CV R2={R['rf_cv_r2']:.3f} (MAE {R['rf_cv_mae']:.1f}) | forward-chaining R2={R['rf_fc_r2']:.3f}")

resid = y - oof                                   # out-of-fold residuals of the calibrated model
RQ = {k_: float(np.quantile(resid, v_)) for k_, v_ in dict(q05=.05, q25=.25, q75=.75, q95=.95).items()}
R["resid_q"] = RQ
print("OOF residual quantiles:", {k_: round(v_, 3) for k_, v_ in RQ.items()})

calibrated linear model: shuffled 5-fold CV R2=0.352 (MAE 27.6) | forward-chaining R2=-0.590 (MAE 35.1)
random-forest benchmark: shuffled CV R2=0.658 (MAE 19.3) | forward-chaining R2=0.124
OOF residual quantiles: {'q05': -61.579, 'q25': -22.195, 'q75': 17.683, 'q95': 66.306}


random-forest benchmark: shuffled CV R2=0.658 (MAE 19.3) | forward-chaining R2=0.124
OOF residual quantiles: {'q05': -61.579, 'q25': -22.195, 'q75': 17.683, 'q95': 66.306}


## 3. Dissatisfaction (defect-proxy) classifier, CV grouped by order

**What happens here.** A gradient-boosting classifier predicts dissatisfaction from price, freight value and delivery delay; folds are grouped by order so that lines of one order never sit in both training and test data. Its macro-F1 is only used as a *skill score* for the fuzzy heights. The fuzzy defective fraction itself comes from the observed share `theta_hat` with a Beta-Binomial predictive interval (`TQ`).

In [6]:
Xo = Ln[["price", "freight_value", "delay_days"]].values; yo = Ln["dissat"].values; grp = Ln["order_id"].values
clf = HistGradientBoostingClassifier(random_state=SEED)
skf = StratifiedGroupKFold(10, shuffle=True, random_state=SEED)
proba = cross_val_predict(clf, Xo, yo, cv=skf, groups=grp, method="predict_proba")[:, 1]
mf1 = lambda p_, t: f1_score(yo, (p_ >= t).astype(int), average="macro")
best = max(np.arange(0.2, 0.8, 0.01), key=lambda t: mf1(proba, t))
p2 = cross_val_predict(clf, Xo[:, :2], yo, cv=skf, groups=grp, method="predict_proba")[:, 1]
R.update(clf_auc=float(roc_auc_score(yo, proba)), clf_f1=float(mf1(proba, .5)), clf_thr=float(best), clf_f1_best=float(mf1(proba, best)),
         clf_auc_nodelay=float(roc_auc_score(yo, p2)), clf_f1_nodelay=float(mf1(p2, .5)))
print(f"AUC={R['clf_auc']:.3f}  macro-F1@0.5={R['clf_f1']:.3f}  macro-F1@{best:.2f}={R['clf_f1_best']:.3f}"
      f" | without delay: AUC={R['clf_auc_nodelay']:.3f}, macro-F1={R['clf_f1_nodelay']:.3f}")

# Beta-Binomial predictive interval for the dissatisfied share of a future batch of 47 units
k, n_ = R["k_def"], R["n_def"]; theta_hat = R["theta_hat"]
rng = np.random.default_rng(SEED)
th_draw = rng.beta(k + 1, n_ - k + 1, 200000); frac = rng.binomial(47, th_draw) / 47
TQ = {c_: float(np.quantile(frac, p_)) for c_, p_ in dict(q05=.05, q25=.25, q75=.75, q95=.95).items()}
R["theta_bb"] = TQ
print("theta_hat = %.4f | Beta-Binomial predictive quantiles (batch of 47):" % theta_hat, {c_: round(v_, 4) for c_, v_ in TQ.items()})

AUC=0.673  macro-F1@0.5=0.658  macro-F1@0.29=0.668 | without delay: AUC=0.556, macro-F1=0.475
theta_hat = 0.1300 | Beta-Binomial predictive quantiles (batch of 47): {'q05': 0.0638, 'q25': 0.0851, 'q75': 0.1702, 'q95': 0.2128}


## 4. Model: crisp and fuzzy TPU(Q, B, P)

`tpu` is Eq. (tpu); `crisp_alt` is Algorithm 1; `M_tpu` is the defuzzified objective, Eq. (MTPU);
`fuzzy_B_closed` / `fuzzy_P_closed` are Eqs. (Bstarfuzzy) / (Pstarfuzzy); `fuzzy_numeric` maximizes the profile over Q.
The price is restricted to `P <= P_CAP` (the highest weekly average price observed); the profit is concave in P, so the capped optimum is the clipped closed form.

| paper | code |
|---|---|
| demand `D(P, tau)` | `D_fun` |
| profit per unit time `TPU(Q, B, P)`, Eq. (tpu) | `tpu` |
| Algorithm 1 (alternating optimization, price capped) | `crisp_alt` |
| defuzzified profit `M(TPU)_{m,n}`, Eq. (MTPU) | `M_tpu` |
| fuzzy `B*` and `P*`, Eqs. (Bstarfuzzy), (Pstarfuzzy) | `fuzzy_B_closed`, `fuzzy_P_closed` |
| fuzzy optimum (search over Q) | `fuzzy_numeric` |
| Definition 1 (intervals to fuzzy sets) | `build_F` |

In [7]:
def D_fun(P, a0, a1, a2, tau): return a0 - a1 * P + a2 * tau

def tpu(Q, B, P, p, D=None, q=None):
    q = p["q"] if q is None else q
    D = D_fun(P, p["a0"], p["a1"], p["a2"], p["tau"]) if D is None else D
    phi = (P + p["v"] * (1 - q) / q - p["K"] / (q * Q) - p["c"] / q - p["d"] / q
           - p["h"] * (1 - q) * Q / (2 * p["x"] * q))
    return p["h"] * B - p["h"] * q * Q / 2 - (p["b"] + p["h"]) * B**2 / (2 * q * Q) + D * phi

def crisp_alt(p, P0=45.0, eps=1e-9, itmax=500, cap=np.inf):
    P = P0
    for it in range(itmax):
        q, h, b, x, K = p["q"], p["h"], p["b"], p["x"], p["K"]
        D = D_fun(P, p["a0"], p["a1"], p["a2"], p["tau"])
        Q = np.sqrt(2 * K * D / (h * (1 - q) * D / x + h * q**2 * b / (b + h)))
        B = h * q * Q / (b + h)
        Pn = ((p["a0"] + p["a2"] * p["tau"]) / (2 * p["a1"]) + K / (2 * Q * q) + (p["c"] + p["d"]) / (2 * q)
              - p["v"] * (1 - q) / (2 * q) + Q * h * (1 - q) / (4 * x * q))
        Pn = min(Pn, cap)
        if abs(Pn - P) < eps: P = Pn; break
        P = Pn
    return Q, B, P, tpu(Q, B, P, p), it + 1

def crisp_numeric(p, cap=np.inf):
    ub = cap if np.isfinite(cap) else 1e4
    best = None
    for P0 in [50, 100, 150, 190]:
        r = minimize(lambda z: -tpu(z[0], z[1], z[2], p), [40, 8, min(P0, ub)], method="Powell",
                     bounds=[(1, 1e5), (0, 1e5), (1, ub)], options=dict(xtol=1e-10, ftol=1e-12, maxiter=50000))
        if best is None or r.fun < best.fun: best = r
    return best.x, -best.fun

def fuzzy_aggregates(F):
    a0L, a0U, qL, qU = map(np.array, (F["a0L"], F["a0U"], F["qL"], F["qU"]))
    return dict(QL=np.sum(1 / qL) / 8, QU=np.sum(1 / qU) / 8, qLs=np.sum(qL) / 8, qUs=np.sum(qU) / 8,
                a0Lbar=np.sum(a0L) / 8, a0Ubar=np.sum(a0U) / 8,
                Z0L=np.sum(a0L / qL[::-1]) / 8, Z0U=np.sum(a0U / qU[::-1]) / 8)

def M_tpu(Q, B, P, p, F, m, n):
    g = fuzzy_aggregates(F); lin = -p["a1"] * P + p["a2"] * p["tau"]
    DL = g["a0Lbar"] + lin * 4 / 8; DU = g["a0Ubar"] + lin * 4 / 8
    ZL = g["Z0L"] + lin * g["QL"]; ZU = g["Z0U"] + lin * g["QU"]
    h, b, x, K = p["h"], p["b"], p["x"], p["K"]
    return (h * B - h * Q / 2 * (g["qLs"] * m + g["qUs"] * n)
            - (b + h) * B**2 / (2 * Q) * (g["QL"] * m + g["QU"] * n)
            + (P - p["v"] + h * Q / (2 * x)) * (DL * m + DU * n)
            - (p["c"] + p["d"] - p["v"] + K / Q + h * Q / (2 * x)) * (ZL * m + ZU * n))

def fuzzy_B_closed(Q, p, F, m, n):
    g = fuzzy_aggregates(F); return Q * p["h"] / ((p["b"] + p["h"]) * (g["QL"] * m + g["QU"] * n))

def fuzzy_P_closed(Q, p, F, m, n):
    g = fuzzy_aggregates(F); K, c, d, v, h, x = p["K"], p["c"], p["d"], p["v"], p["h"], p["x"]
    W = g["QL"] * m + g["QU"] * n
    return (K * W / (Q * (m + n)) + W * (c + d - v) / (m + n) + v / 2
            + (g["a0Lbar"] * m + g["a0Ubar"] * n + p["a2"] * p["tau"] * (m + n) / 2) / (p["a1"] * (m + n))
            + Q * h * (2 * W - (m + n)) / (4 * x * (m + n)))

def fuzzy_numeric(p, F, m, n, cap=np.inf, Qmax=5000.0):
    # profile optimization: closed-form B*(Q), P*(Q) (price clipped at the cap), then a 1-D search over Q
    Pq = lambda Q: min(fuzzy_P_closed(Q, p, F, m, n), cap)
    prof = lambda Q: -M_tpu(Q, fuzzy_B_closed(Q, p, F, m, n), Pq(Q), p, F, m, n)
    r = minimize_scalar(prof, bounds=(5.0, Qmax), method="bounded", options=dict(xatol=1e-10))
    Q = r.x
    if Q > 0.99 * Qmax: print(f"  WARNING: unbounded in Q for (m,n)=({m},{n}); Q hit {Qmax}")
    return np.array([Q, fuzzy_B_closed(Q, p, F, m, n), Pq(Q)]), -r.fun

def build_F(a0_, r_, th_hat, thq):
    qh = 1 - th_hat
    return dict(a0L=[a0_ + r_["q25"], a0_, a0_, a0_ + r_["q75"]], a0U=[a0_ + r_["q05"], a0_, a0_, a0_ + r_["q95"]],
                qL=[1 - thq["q75"], qh, qh, 1 - thq["q25"]], qU=[1 - thq["q95"], qh, qh, 1 - thq["q05"]])

## 5. Run: crisp optimum, fuzzy optimum, checks

All quantities are **per week** (h, b in \$/unit/week; x in units/week). Cost parameters are illustrative: the dataset has no cost data.

**What happens here.** The crisp optimum is solved without and with the price cap (highest observed weekly price). The fuzzy sets are built from `RQ` and from the Beta-Binomial interval for a batch of the ordered size, the heights `m`, `n` are set from the cross-validated skills, and the fuzzy optimum is found. Three checks follow: (1) with degenerate fuzzy sets the fuzzy model returns the crisp solution, (2) a direct 3-D search agrees with the closed forms, (3) the concavity condition and Hessian are evaluated.

In [8]:
P_CAP = R["price_max"]
p = dict(tau=40, K=100, c=30, v=8, h=5 / 52, b=15 / 52, d=0.5, x=15000 / 52, a0=a0, a1=a1, a2=a2, q=1 - theta_hat)
R["P_cap"] = P_CAP

# unconstrained price (reference) and capped price
Qu, Bu, Pu, Tu, itu = crisp_alt(p)
Qc, Bc, Pc, Tc, it = crisp_alt(p, cap=P_CAP)
zc, Tn = crisp_numeric(p, cap=P_CAP)
Dc = D_fun(Pc, a0, a1, a2, 40)
R.update(crisp_unc=[Qu, Bu, Pu, Tu], crisp=[Qc, Bc, Pc, Tc], crisp_D=float(Dc), crisp_iter=it)
print(f"crisp, price unrestricted : Q*={Qu:.3f}  B*={Bu:.3f}  P*={Pu:.3f}  TPU*={Tu:.2f}   <- far above the observed prices")
print(f"crisp, P <= {P_CAP:.1f} ({it} it): Q*={Qc:.3f}  B*={Bc:.3f}  P*={Pc:.3f}  TPU*={Tc:.2f}  D={Dc:.3f}")
print(f"crisp, direct search      : Q*={zc[0]:.3f}  B*={zc[1]:.3f}  P*={zc[2]:.3f}  TPU*={Tn:.2f}")

# defect interval for a batch of the size actually ordered: recompute the Beta-Binomial predictive quantiles for Q = round(crisp Q*)
BATCH = int(round(Qc)); R["batch"] = BATCH
frac = np.random.default_rng(SEED).binomial(BATCH, th_draw) / BATCH
TQ = {c_: float(np.quantile(frac, p_)) for c_, p_ in dict(q05=.05, q25=.25, q75=.75, q95=.95).items()}
R["theta_bb"] = TQ
print(f"Beta-Binomial predictive quantiles for the ordered batch (n={BATCH}):", {c_: round(v_, 4) for c_, v_ in TQ.items()})
F = build_F(a0, RQ, theta_hat, TQ)
R["F"] = {k_: [float(v_) for v_ in vv] for k_, vv in F.items()}
print("a0 breakpoints  L:", np.round(F["a0L"], 3), " U:", np.round(F["a0U"], 3))
print("q  breakpoints  L:", np.round(F["qL"], 4), " U:", np.round(F["qU"], 4))

hU_d = 0.5 + 0.5 * max(R["cv_r2"], 0.0); hU_q = 0.5 + 0.5 * R["clf_f1"]
m = round(min(0.8 * hU_d, 0.8 * hU_q), 3); n = round(min(hU_d, hU_q), 3)
R.update(hU_d=hU_d, hU_q=hU_q, hL_d=0.8 * hU_d, hL_q=0.8 * hU_q, m=m, n=n)
print(f"heights: hD^U={hU_d:.3f} hq^U={hU_q:.3f}  ->  m={m}, n={n}")
zf, Mf = fuzzy_numeric(p, F, m, n, cap=P_CAP)
zfu, Mfu = fuzzy_numeric(p, F, m, n)
R.update(fuzzy=[*map(float, zf), float(Mf)], fuzzy_unc=[*map(float, zfu), float(Mfu)])
print(f"fuzzy, price unrestricted: Q*={zfu[0]:.3f}  B*={zfu[1]:.3f}  P*={zfu[2]:.3f}")
print(f"fuzzy, P <= {P_CAP:.1f}   : Q*={zf[0]:.3f}  B*={zf[1]:.3f}  P*={zf[2]:.3f}  M(TPU)={Mf:.2f}")
print(f"change vs crisp: Q {100*(zf[0]/Qc-1):+.1f}%  B {100*(zf[1]/Bc-1):+.1f}%  P {100*(zf[2]/Pc-1):+.1f}%  (M(TPU) is not comparable with TPU)")

# ---- checks ----
zd, Md = fuzzy_numeric(p, dict(a0L=[a0]*4, a0U=[a0]*4, qL=[p['q']]*4, qU=[p['q']]*4), 1, 1, cap=P_CAP)
print(f"check 1: degenerate fuzzy sets, m=n=1 -> Q={zd[0]:.3f} B={zd[1]:.3f} P={zd[2]:.3f}   (crisp: {Qc:.3f} {Bc:.3f} {Pc:.3f})")
f3 = lambda z: -M_tpu(z[0], z[1], z[2], p, F, m, n)
r3 = minimize(f3, zf, method="Powell", bounds=[(1, 1e5), (0, 1e5), (1, P_CAP)], options=dict(xtol=1e-10, ftol=1e-12, maxiter=50000))
print(f"check 2: direct 3-D search of the fuzzy objective -> Q={r3.x[0]:.3f} B={r3.x[1]:.3f} P={r3.x[2]:.3f}")

def hess(z, free, e=1e-3):
    f = lambda a: tpu(a[0], a[1], a[2], p); z = np.array(z, float); s = np.maximum(np.abs(z), 1) * e
    Hm = np.zeros((len(free), len(free)))
    for ii, i in enumerate(free):
        for jj, j in enumerate(free):
            def sh(si, sj):
                w = z.copy(); w[i] += si * s[i]; w[j] += sj * s[j]; return f(w)
            Hm[ii, jj] = (sh(1, 1) - sh(1, -1) - sh(-1, 1) + sh(-1, -1)) / (4 * s[i] * s[j])
    return Hm
bind = Pc >= P_CAP - 1e-6
free = [0, 1] if bind else [0, 1, 2]
ev = np.linalg.eigvalsh(hess([Qc, Bc, Pc], free))
R.update(price_cap_binds=bool(bind), hess_eig=[float(v_) for v_ in ev])
print(f"check 3: price cap {'BINDS' if bind else 'does not bind'}; Hessian of TPU in {'(Q,B)' if bind else '(Q,B,P)'} at the optimum, eigenvalues: {np.round(ev, 5)}")
q_, K_, c_, d_, v_, h_, x_ = p["q"], p["K"], p["c"], p["d"], p["v"], p["h"], p["x"]
Mden = 12*K_**2*x_**2 + 8*K_*Qu*x_**2*(c_ + d_ - v_*(1 - q_)) + Qu**4*h_**2*(1 - q_)**2
R["thm_threshold_unc"] = float(8*K_*Qu*q_*x_**2*(a0 + a2*40) / Mden)
print(f"         Theorem 1 threshold at the unrestricted optimum: a1={a1:.4f} vs {R['thm_threshold_unc']:.4f} -> {'holds' if a1 < R['thm_threshold_unc'] else 'FAILS'}")

crisp, price unrestricted : Q*=434.187  B*=94.433  P*=290.088  TPU*=13715.22   <- far above the observed prices
crisp, P <= 194.3 (2 it): Q*=505.148  B*=109.866  P*=194.293  TPU*=11792.47  D=73.714
crisp, direct search      : Q*=505.148  B*=109.866  P*=194.293  TPU*=11792.47
Beta-Binomial predictive quantiles for the ordered batch (n=505): {'q05': 0.105, 'q25': 0.1188, 'q75': 0.1406, 'q95': 0.1564}
a0 breakpoints  L: [45.378 67.574 67.574 85.257]  U: [  5.994  67.574  67.574 133.88 ]
q  breakpoints  L: [0.8594 0.87   0.87   0.8812]  U: [0.8436 0.87   0.87   0.895 ]
heights: hD^U=0.676 hq^U=0.829  ->  m=0.541, n=0.676
fuzzy, price unrestricted: Q*=644.243  B*=230.175  P*=290.422
fuzzy, P <= 194.3   : Q*=742.021  B*=265.109  P*=194.293  M(TPU)=7187.56
change vs crisp: Q +46.9%  B +141.3%  P +0.0%  (M(TPU) is not comparable with TPU)
check 1: degenerate fuzzy sets, m=n=1 -> Q=505.148 B=109.866 P=194.293   (crisp: 505.148 109.866 194.293)
check 2: direct 3-D search of the fuzzy objective -

## 6. Paired Monte Carlo: realized profit of the crisp vs the fuzzy policy

**What happens here.** 20,000 weeks are simulated. In each week demand is the model demand plus a draw from the out-of-fold residuals, and the defective share is a Beta-Binomial draw. Both policies face *the same* draws (paired comparison), so the difference in realized profit is not blurred by luck.

In [9]:
NSIM = 20000
rng0 = np.random.default_rng(7)
eps_d = rng0.choice(resid, NSIM); thb = rng0.beta(k + 1, n_ - k + 1, NSIM)      # common random numbers for both policies
def realized(Q, B, P, seed):
    r_ = np.random.default_rng(seed)
    D = np.clip(D_fun(P, a0, a1, a2, 40) + eps_d, 1.0, None)
    th = r_.binomial(int(round(Q)), thb) / int(round(Q)); qq = np.clip(1 - th, 0.5, 1.0)
    return np.array([tpu(Q, B, P, p, D=D[i], q=qq[i]) for i in range(NSIM)])
pa = realized(Qc, Bc, Pc, 7); pb = realized(*zf, 7)
diff = pb - pa
bs = np.random.default_rng(3).choice(diff, (500, NSIM)).mean(axis=1)
R["mc"] = dict(crisp_mean=float(pa.mean()), fuzzy_mean=float(pb.mean()), crisp_p5=float(np.quantile(pa, .05)), fuzzy_p5=float(np.quantile(pb, .05)),
               diff=float(diff.mean()), diff_pct=float(100 * diff.mean() / pa.mean()), ci=[float(np.quantile(bs, .025)), float(np.quantile(bs, .975))],
               p_t=float(stats.ttest_rel(pb, pa).pvalue), p_w=float(stats.wilcoxon(diff).pvalue), share=float(np.mean(diff > 0)))
mc = R["mc"]
print(f"mean realized TPU: crisp={mc['crisp_mean']:.1f}  fuzzy={mc['fuzzy_mean']:.1f}  | 5th percentile: crisp={mc['crisp_p5']:.1f} fuzzy={mc['fuzzy_p5']:.1f}")
print(f"paired diff (fuzzy-crisp)={mc['diff']:.2f} ({mc['diff_pct']:.2f}%)  95% CI [{mc['ci'][0]:.2f}, {mc['ci'][1]:.2f}]"
      f"  paired t p={mc['p_t']:.2g}  Wilcoxon p={mc['p_w']:.2g}  share(fuzzy>crisp)={mc['share']:.3f}")

mean realized TPU: crisp=11920.9  fuzzy=11915.5  | 5th percentile: crisp=1781.9 fuzzy=1771.7
paired diff (fuzzy-crisp)=-5.42 (-0.05%)  95% CI [-6.09, -4.75]  paired t p=7.7e-55  Wilcoxon p=7.1e-96  share(fuzzy>crisp)=0.409


## 7. Tables for the paper (theta, K, price sensitivity, heights)

**What happens here.** The crisp optimum is recomputed while one input is varied (defective fraction, ordering cost, price cap, elasticity), and the fuzzy optimum for different heights. Each table is printed and stored in `results.json`.

In [10]:
def solve(cap=P_CAP, **kw):
    pp = dict(p); pp.update(kw)
    if "theta" in kw: pp["q"] = 1 - kw["theta"]
    return crisp_alt(pp, cap=cap)[:4]

T = {}
T["theta"] = pd.DataFrame([(t_, *solve(theta=t_)) for t_ in [0.05, 0.09, round(theta_hat, 4), 0.17, 0.20]], columns=["theta", "Q*", "B*", "P*", "TPU*"])
T["K"] = pd.DataFrame([(K__, *solve(K=K__)) for K__ in [50, 75, 100, 125, 150]], columns=["K", "Q*", "B*", "P*", "TPU*"])
rows = []
for e_ in [eps_hat - se_eps, eps_hat, eps_hat + se_eps]:          # +-1 clustered s.e. around the elasticity
    a1_ = -e_ * Dbar / Pbar; a0_, a2_ = calibrate(a1_)
    pp = dict(p, a0=a0_, a1=a1_, a2=a2_)
    Qa, Ba, Pa, Ta, _ = crisp_alt(pp, cap=P_CAP); Qb, Bb, Pb, Tb, _ = crisp_alt(pp)
    rows.append((e_, a1_, Qa, Ba, Pa, Ta, Pb))
T["elasticity"] = pd.DataFrame(rows, columns=["elasticity", "a1", "Q*", "B*", "P*(capped)", "TPU*", "P*(unrestricted)"])
rows = []
for cp in [R["price_p90"], 150.0, P_CAP, 250.0, np.inf]:
    Qa, Ba, Pa, Ta, _ = crisp_alt(p, cap=cp); rows.append(("none" if np.isinf(cp) else round(cp, 1), Qa, Ba, Pa, Ta))
T["cap"] = pd.DataFrame(rows, columns=["price cap", "Q*", "B*", "P*", "TPU*"])
rows = []
for mm, nn in [(0.4, 0.5), (m, n), (0.6, 0.75), (0.8, 1.0), (1.0, 1.0)]:
    z, Mv = fuzzy_numeric(p, F, mm, nn, cap=P_CAP); rows.append((mm, nn, *z))
T["heights"] = pd.DataFrame(rows, columns=["m", "n", "Q*", "B*", "P*"])
for name, df in T.items():
    print(f"--- sensitivity: {name} ---"); print(df.round(3).to_string(index=False)); print()
    R["T_" + name] = df.round(4).values.tolist()

--- sensitivity: theta ---
 theta      Q*      B*      P*      TPU*
  0.05 471.512 111.984 194.293 11953.698
  0.09 487.910 111.000 194.293 11876.665
  0.13 505.136 109.867 194.293 11792.524
  0.17 523.201 108.564 194.293 11700.247
  0.20 537.299 107.460 194.293 11624.962

--- sensitivity: K ---
  K      Q*      B*      P*      TPU*
 50 357.193  77.687 194.293 11802.293
 75 437.471  95.147 194.293 11796.961
100 505.148 109.866 194.293 11792.467
125 564.772 122.834 194.293 11788.507
150 618.677 134.558 194.293 11784.927

--- sensitivity: elasticity ---
 elasticity    a1      Q*      B*  P*(capped)      TPU*  P*(unrestricted)
     -0.396 0.339 469.383 102.088     194.293 10098.800           207.360
     -0.245 0.210 505.148 109.866     194.293 11792.467           290.088
     -0.094 0.080 538.033 117.019     194.293 13486.288           639.609

--- sensitivity: cap ---
price cap      Q*      B*      P*      TPU*
    134.0 544.132 118.345 134.041  8613.051
    150.0 534.153 116.175 150.00

## 8. Signal tests (price and delivery delay vs. the dissatisfaction proxy)

**What happens here.** Checks whether price and delivery delay differ between satisfied and dissatisfied lines (Welch tests and one-way ANOVA across review scores). Delay does; price does not.

In [11]:
for col in ["price", "freight_value", "delay_days"]:
    a_ = Ln.loc[Ln.dissat == 1, col]; b_ = Ln.loc[Ln.dissat == 0, col]
    print(f"{col:14s}: dissatisfied mean={a_.mean():8.2f}  satisfied mean={b_.mean():8.2f}  Welch p={stats.ttest_ind(a_, b_, equal_var=False).pvalue:.3g}")
    R["sig_" + col] = [float(a_.mean()), float(b_.mean())]
rs_ = Ln["review_score"].round().astype(int)
Fp = stats.f_oneway(*[Ln.loc[rs_ == s_, "price"] for s_ in sorted(rs_.unique())])
Fd = stats.f_oneway(*[Ln.loc[rs_ == s_, "delay_days"] for s_ in sorted(rs_.unique())])
R.update(anova_price=[float(Fp.statistic), float(Fp.pvalue)], anova_delay=[float(Fd.statistic), float(Fd.pvalue)])
print("ANOVA price across review scores: F=%.2f p=%.3g" % tuple(R["anova_price"]))
print("ANOVA delay across review scores: F=%.2f p=%.3g" % tuple(R["anova_delay"]))

json.dump(R, open(f"{OUT_DIR}/results.json", "w"), indent=1, default=float)
print("\nsaved", f"{OUT_DIR}/results.json")

price         : dissatisfied mean=  121.76  satisfied mean=  111.83  Welch p=0.096
freight_value : dissatisfied mean=   21.41  satisfied mean=   19.07  Welch p=0.00026
delay_days    : dissatisfied mean=   -4.81  satisfied mean=  -12.30  Welch p=2.72e-56
ANOVA price across review scores: F=1.73 p=0.14
ANOVA delay across review scores: F=183.59 p=4.5e-151

saved outputs/results.json


## 9. Benchmarks and statistical tests

Seven policies are compared by the **realized weekly profit** of Eq. (tpu) in the same 20,000 simulated weeks (common random numbers):

| code | policy | price | fuzziness |
|---|---|---|---|
| B1 | classical crisp (Salameh-Jaber / Wee et al. type) | fixed at the mean observed price | none |
| B2 | type-1 fuzzy (Chang type), data-derived 90% trapezoid | fixed | type-1 |
| B3 | interval type-2 fuzzy with expert-set spreads (Kumar type) | fixed | type-2, expert breakpoints |
| B4 | proposed crisp | optimized, capped at the observed maximum | none |
| B5 | **proposed fuzzy**: data-derived interval type-2 | optimized, capped | type-2, data-derived |
| B6 | ablation: type-1 fuzzy, data-derived | optimized, capped | type-1 |
| B7 | ablation: interval type-2 with expert-set spreads | optimized, capped | type-2, expert |

Expert-set spreads (assumption, not from data): demand intercept +-10% / +-20% of mean weekly demand, defective fraction +-20% / +-40% of its estimate, heights m=0.8, n=1.
The benchmarks are *adapted implementations* of the cited model types in our setting, not the authors' original code.
Each comparison is against B5; p-values are Holm-adjusted over the six comparisons. With 20,000 paired draws p-values are tiny, so the effect sizes (dollars and percent per week) matter more.

In [12]:
from scipy.optimize import minimize_scalar
Pfix = R["price_mean"]; Dbar_ = float(np.mean(y)); qh_ = 1 - theta_hat
def holm(pv):
    pv = np.asarray(pv); o = np.argsort(pv); adj = np.empty_like(pv); run = 0.0
    for r_, i_ in enumerate(o): run = max(run, (len(pv) - r_) * pv[i_]); adj[i_] = min(1.0, run)
    return adj
def F_t1_from(a0_, rq_, tq_):                      # type-1 trapezoid: the 90% members only
    q_ = 1 - theta_hat
    return dict(a0L=[a0_ + rq_["q05"], a0_, a0_, a0_ + rq_["q95"]], a0U=[a0_ + rq_["q05"], a0_, a0_, a0_ + rq_["q95"]],
                qL=[1 - tq_["q95"], q_, q_, 1 - tq_["q05"]], qU=[1 - tq_["q95"], q_, q_, 1 - tq_["q05"]])
def F_exp_from(a0_, Dref_):                        # expert-set spreads
    q_ = 1 - theta_hat
    return dict(a0L=[a0_ - .1 * Dref_, a0_, a0_, a0_ + .1 * Dref_], a0U=[a0_ - .2 * Dref_, a0_, a0_, a0_ + .2 * Dref_],
                qL=[1 - 1.2 * theta_hat, q_, q_, 1 - .8 * theta_hat], qU=[1 - 1.4 * theta_hat, q_, q_, 1 - .6 * theta_hat])
M_EXP, N_EXP = 0.8, 1.0
def crisp_fixedP(pp, P):
    D = max(D_fun(P, pp["a0"], pp["a1"], pp["a2"], pp["tau"]), 1.0); q_, h_, b_, x_, K_ = pp["q"], pp["h"], pp["b"], pp["x"], pp["K"]
    Q = np.sqrt(2 * K_ * D / (h_ * (1 - q_) * D / x_ + h_ * q_**2 * b_ / (b_ + h_))); return Q, h_ * q_ * Q / (b_ + h_), P
def fuzzy_fixedP(pp, Fz, mm, nn, P):
    f_ = lambda Q: -M_tpu(Q, fuzzy_B_closed(Q, pp, Fz, mm, nn), P, pp, Fz, mm, nn)
    Q = minimize_scalar(f_, bounds=(5.0, 5000.0), method="bounded", options=dict(xatol=1e-8)).x
    return Q, fuzzy_B_closed(Q, pp, Fz, mm, nn), P
F_t1 = F_t1_from(a0, RQ, TQ); F_ex = F_exp_from(a0, Dbar_)
pol = {
 "B1 classical crisp, fixed price":        crisp_fixedP(p, Pfix),
 "B2 type-1 fuzzy, fixed price":           fuzzy_fixedP(p, F_t1, 1, 1, Pfix),
 "B3 IT2 expert-set, fixed price":         fuzzy_fixedP(p, F_ex, M_EXP, N_EXP, Pfix),
 "B4 proposed crisp (price optimized)":    (Qc, Bc, Pc),
 "B5 proposed fuzzy (data-derived IT2)":   tuple(zf),
 "B6 type-1 fuzzy (price optimized)":      tuple(fuzzy_numeric(p, F_t1, 1, 1, cap=P_CAP)[0]),
 "B7 IT2 expert-set (price optimized)":    tuple(fuzzy_numeric(p, F_ex, M_EXP, N_EXP, cap=P_CAP)[0]),
}
def sim_profit(Q, B, P, seed=7):
    D = np.clip(D_fun(P, a0, a1, a2, 40) + eps_d, 1.0, None)
    th = np.random.default_rng(seed).binomial(int(round(Q)), thb) / int(round(Q))
    return tpu(Q, B, P, p, D=D, q=np.clip(1 - th, 0.5, 1.0))
S = {nm: sim_profit(*z) for nm, z in pol.items()}
ref = "B5 proposed fuzzy (data-derived IT2)"; cvar = lambda v: float(v[v <= np.quantile(v, .05)].mean())
names = [nm for nm in pol if nm != ref]
pt = [stats.ttest_rel(S[ref], S[nm]).pvalue for nm in names]; pw = [stats.wilcoxon(S[ref] - S[nm]).pvalue for nm in names]
ph = holm(pt); phw = holm(pw); bsr = np.random.default_rng(11); rows = []
for nm, z in pol.items():
    r_ = dict(policy=nm, Q=z[0], B=z[1], P=z[2], mean=S[nm].mean(), p5=float(np.quantile(S[nm], .05)), cvar5=cvar(S[nm]))
    if nm != ref:
        d_ = S[ref] - S[nm]; bs_ = bsr.choice(d_, (500, len(d_))).mean(1); i_ = names.index(nm)
        r_.update(diff=d_.mean(), diff_pct=100 * d_.mean() / S[nm].mean(), lo=float(np.quantile(bs_, .025)), hi=float(np.quantile(bs_, .975)),
                  p_holm_t=float(ph[i_]), p_holm_w=float(phw[i_]), share=float((d_ > 0).mean()))
    rows.append(r_)
BN = pd.DataFrame(rows); R["bench_sim"] = BN.to_dict("records")
print("Simulated realized weekly profit; diff = B5 minus the policy in the row (positive: proposed fuzzy better)")
print(BN.round(2).to_string(index=False))

Simulated realized weekly profit; diff = B5 minus the policy in the row (positive: proposed fuzzy better)
                              policy      Q      B      P     mean      p5   cvar5    diff  diff_pct      lo      hi  p_holm_t  p_holm_w  share
     B1 classical crisp, fixed price 557.27 121.20 112.46  7142.67 2204.18 1141.97 4772.82     66.82 4731.43 4812.44       0.0       0.0   0.94
        B2 type-1 fuzzy, fixed price 562.17 122.17 112.46  7142.59 2203.64 1149.13 4772.91     66.82 4732.00 4812.52       0.0       0.0   0.94
      B3 IT2 expert-set, fixed price 579.32 139.83 112.46  7142.40 2205.03 1145.97 4773.10     66.83 4730.46 4809.12       0.0       0.0   0.94
 B4 proposed crisp (price optimized) 505.15 109.87 194.29 11920.92 1781.89  561.92   -5.42     -0.05   -6.04   -4.77       0.0       0.0   0.41
B5 proposed fuzzy (data-derived IT2) 742.02 265.11 194.29 11915.50 1771.74  545.55     NaN       NaN     NaN     NaN       NaN       NaN    NaN
   B6 type-1 fuzzy (price opti

In [13]:
Pw = wk.avg_price.values; T = wk.t_obs.values
def backtest(pc, a1_=a1):
    out = []
    for t_ in range(30, len(wk)):                                   # origin t: train on weeks < t, decide for week t at its observed price
        idx = np.arange(t_); yad = y[idx] + a1_ * (Pw[idx] - Pw[idx].mean()); X_ = np.column_stack([np.ones(len(idx)), T[idx]])
        (al, a2t), *_ = np.linalg.lstsq(X_, yad, rcond=None); a0t = al + a1_ * Pw[idx].mean()
        res = y[idx] - (a0t - a1_ * Pw[idx] + a2t * T[idx]); rq_ = {k_: float(np.quantile(res, v_)) for k_, v_ in dict(q05=.05, q25=.25, q75=.75, q95=.95).items()}
        pt_ = dict(pc, a0=a0t, a1=a1_, a2=a2t, tau=int(T[t_])); P_ = Pw[t_]; Dh = max(a0t - a1_ * P_ + a2t * T[t_], 1.0)
        z = {"crisp": crisp_fixedP(dict(pt_), P_), "t1": fuzzy_fixedP(pt_, F_t1_from(a0t, rq_, TQ), 1, 1, P_),
             "it2": fuzzy_fixedP(pt_, build_F(a0t, rq_, theta_hat, TQ), m, n, P_), "exp": fuzzy_fixedP(pt_, F_exp_from(a0t, float(y[idx].mean())), M_EXP, N_EXP, P_)}
        z["oracle"] = crisp_fixedP(dict(pt_, a0=y[t_] + a1_ * P_ - a2t * T[t_]), P_)       # knows this week's demand
        out.append([tpu(*z[k_], pt_, D=y[t_]) for k_ in ["crisp", "t1", "it2", "exp", "oracle"]])
    return np.array(out)
BT = backtest(p); lab = ["classical crisp", "type-1 fuzzy (data)", "IT2 data-derived (proposed)", "IT2 expert-set", "oracle"]
others = [0, 1, 3]; pw_ = holm([stats.wilcoxon(BT[:, 2] - BT[:, i]).pvalue for i in others]); rows = []
for j, nm in enumerate(lab):
    r_ = dict(policy=nm, mean=BT[:, j].mean(), p5=float(np.quantile(BT[:, j], .05)))
    if j in others:
        d_ = BT[:, 2] - BT[:, j]; bs_ = np.random.default_rng(1).choice(d_, (2000, len(d_))).mean(1)
        r_.update(diff=d_.mean(), lo=float(np.quantile(bs_, .025)), hi=float(np.quantile(bs_, .975)), p_holm=float(pw_[others.index(j)]), share=float((d_ > 0).mean()))
    rows.append(r_)
BTT = pd.DataFrame(rows); R["backtest"] = BTT.to_dict("records"); R["backtest_weeks"] = int(BT.shape[0])
print(f"Real-week backtest ({BT.shape[0]} weeks, price = observed price, demand = observed demand); diff = proposed minus the policy in the row")
print(BTT.round(2).to_string(index=False))

Real-week backtest (58 weeks, price = observed price, demand = observed demand); diff = proposed minus the policy in the row
                     policy    mean      p5  diff    lo    hi  p_holm  share
            classical crisp 9494.05 4483.89 -7.91 -8.43 -7.41     0.0    0.0
        type-1 fuzzy (data) 9494.05 4483.86 -7.91 -8.42 -7.41     0.0    0.0
IT2 data-derived (proposed) 9486.14 4472.94   NaN   NaN   NaN     NaN    NaN
             IT2 expert-set 9493.84 4483.33 -7.70 -8.16 -7.25     0.0    0.0
                     oracle 9494.72 4486.06   NaN   NaN   NaN     NaN    NaN


## 10. Robustness: scenario grid and cost regimes

**What happens here.** (a) 60 mis-specification scenarios: the true demand level, defective rate and noise differ from the estimates; the crisp and fuzzy policies are compared in each. (b) 12 cost regimes (holding cost, backorder-to-holding ratio): the real-week backtest and the simulation are repeated, and the "oracle gap" (profit gained by knowing demand) shows how much lot-size decisions matter at all.

In [14]:
# (a) 60 scenarios: demand level shift x defective-rate multiplier x noise scale (policies B4 and B5, price capped)
Dref = D_fun(Pc, a0, a1, a2, 40); rows = []
rr = np.random.default_rng(2024); e0 = rr.choice(resid, NSIM); t0 = rr.beta(k + 1, n_ - k + 1, NSIM)
def sim_s(z, ds, tm, ns_):
    D = np.clip(D_fun(z[2], a0, a1, a2, 40) + ds * Dref + ns_ * e0, 1.0, None); th_ = np.clip(t0 * tm, 0, 0.9)
    qq = np.clip(1 - np.random.default_rng(5).binomial(int(round(z[0])), th_) / int(round(z[0])), 0.5, 1.0)
    return tpu(z[0], z[1], z[2], p, D=D, q=qq)
for ds in [-0.2, -0.1, 0.0, 0.1, 0.2]:
    for tm in [0.5, 1.0, 1.5, 2.0]:
        for ns_ in [1.0, 1.5, 2.0]:
            a_ = sim_s((Qc, Bc, Pc), ds, tm, ns_); b_ = sim_s(tuple(zf), ds, tm, ns_); d_ = b_ - a_
            bs_ = np.random.default_rng(1).choice(d_, (300, NSIM)).mean(1)
            rows.append(dict(ds=ds, tm=tm, ns=ns_, diff=d_.mean(), pct=100 * d_.mean() / a_.mean(), lo=np.quantile(bs_, .025), hi=np.quantile(bs_, .975), cv_diff=cvar(b_) - cvar(a_)))
SC = pd.DataFrame(rows)
R["scen"] = dict(n=len(SC), better=int((SC.lo > 0).sum()), worse=int((SC.hi < 0).sum()), cvar_better=int((SC.cv_diff > 0).sum()),
                 pct_min=float(SC.pct.min()), pct_max=float(SC.pct.max()))
print("scenarios:", R["scen"])

# (b) cost regimes: holding cost (fraction of unit cost per year) x backorder-to-holding ratio; real-week backtest and simulation, B4 vs B5
rows = []
for hr in [0.1, 0.2, 0.4, 0.6]:
    for bh in [3, 10, 30]:
        h_ = hr * 30 / 52; pc = dict(p, h=h_, b=bh * h_); bt_ = backtest(pc); d_ = bt_[:, 2] - bt_[:, 0]
        bsb = np.random.default_rng(1).choice(d_, (1000, len(d_))).mean(1)
        Qa, Ba, Pa, _, _ = crisp_alt(pc, cap=P_CAP); zz, _ = fuzzy_numeric(pc, F, m, n, cap=P_CAP)
        def simc(z):
            D = np.clip(D_fun(z[2], a0, a1, a2, 40) + eps_d, 1.0, None); th_ = np.random.default_rng(5).binomial(int(round(z[0])), thb) / int(round(z[0]))
            return tpu(z[0], z[1], z[2], pc, D=D, q=np.clip(1 - th_, 0.5, 1.0))
        sa_, sb_ = simc((Qa, Ba, Pa)), simc(tuple(zz)); ds_ = sb_ - sa_
        rows.append(dict(hold_rate=hr, b_over_h=bh, oracle_gap=bt_[:, 4].mean() - bt_[:, 0].mean(), bt_diff=d_.mean(),
                         bt_lo=float(np.quantile(bsb, .025)), bt_hi=float(np.quantile(bsb, .975)), sim_diff=ds_.mean(), Qc=Qa, Qf=zz[0]))
CR = pd.DataFrame(rows); R["cost_grid"] = CR.to_dict("records")
print(CR.round(2).to_string(index=False))
print("backtest: proposed fuzzy better in", int((CR.bt_lo > 0).sum()), "and worse in", int((CR.bt_hi < 0).sum()), "of", len(CR), "cost regimes")
json.dump(R, open(f"{OUT_DIR}/results.json", "w"), indent=1, default=float)

scenarios: {'n': 60, 'better': 0, 'worse': 60, 'cvar_better': 2, 'pct_min': -0.07660707039841747, 'pct_max': -0.02310763460455252}
 hold_rate  b_over_h  oracle_gap  bt_diff  bt_lo  bt_hi  sim_diff     Qc     Qf
       0.1         3        0.52    -6.13  -6.54  -5.74     -4.31 652.14 957.94
       0.1        10        0.59    -1.15  -1.27  -1.03     -0.70 595.23 652.52
       0.1        30        0.61    -0.34  -0.39  -0.30     -0.23 577.71 596.26
       0.2         3        0.74    -8.66  -9.24  -8.11     -6.13 461.13 677.37
       0.2        10        0.83    -1.62  -1.80  -1.45     -1.35 420.89 461.40
       0.2        30        0.86    -0.49  -0.55  -0.43     -0.21 408.51 421.62
       0.4         3        1.04   -12.25 -13.07 -11.47     -8.82 326.07 478.97
       0.4        10        1.17    -2.29  -2.54  -2.05     -0.97 297.61 326.26
       0.4        30        1.22    -0.69  -0.77  -0.61     -0.71 288.86 298.13
       0.6         3        1.28   -15.01 -16.01 -14.05    -10.12 266

 hold_rate  b_over_h  oracle_gap  bt_diff  bt_lo  bt_hi  sim_diff     Qc     Qf
       0.1         3        0.52    -6.13  -6.54  -5.74     -4.49 652.14 957.94
       0.1        10        0.59    -1.15  -1.27  -1.03     -0.72 595.23 652.52
       0.1        30        0.61    -0.34  -0.39  -0.30     -0.24 577.71 596.26
       0.2         3        0.74    -8.66  -9.24  -8.11     -6.13 461.13 677.37
       0.2        10        0.83    -1.62  -1.80  -1.45     -1.35 420.89 461.40
       0.2        30        0.86    -0.49  -0.55  -0.43     -0.21 408.51 421.62
       0.4         3        1.04   -12.25 -13.07 -11.47     -8.76 326.07 478.97
       0.4        10        1.17    -2.29  -2.54  -2.05     -0.97 297.61 326.26
       0.4        30        1.22    -0.69  -0.77  -0.61     -0.71 288.86 298.13
       0.6         3        1.28   -15.01 -16.01 -14.05    -10.12 266.24 391.08
       0.6        10        1.44    -2.81  -3.11  -2.51     -1.79 243.00 266.39
       0.6        30        1.49    -0.8